## See "Hard Mining Negatives for Semantic Similarity"

https://www.kaggle.com/code/jithinanievarghese/hard-mining-negatives-for-semantic-similarity#Load-Data-and-preprocess-data

In [126]:
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
 os.getcwd(),
 os.pardir+'/playground')
)
#only add it once
if (PROJECT_ROOT not in sys.path):
 sys.path.append(PROJECT_ROOT)

import utils as ut
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import numpy as np 
import pandas as pd 
import csv

In [127]:
def preprocess_text(text):
    """
    clean white space and lower case the text
    """
    return " ".join(text.split()).lower()

In [129]:
df = pd.read_json('../data/trn.json')

df.drop_duplicates(subset=['anchor', 'positive'], inplace=True)
# df.drop_duplicates(subset=['anchor'], inplace=True)
# df.drop_duplicates(subset=['positive'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.anchor = df['anchor'].apply(lambda x: preprocess_text(x))
df.positive = df['positive'].apply(lambda x: preprocess_text(x))

# df.head()

# Embed the columns of interest

### Get list of positives and anchors

In [130]:
%%time
from sentence_transformers import InputExample
from tqdm.auto import tqdm  # so we see progress bar
def getsentencelists(df,cols):
    '''
    param: df dataframe
    param: cols list of columns in dataframe to return lists from
    return: tuple of lists, each list is a string of all strings in column
     of InputExample objects
     ex.
     cols=['positive','anchor']
    positives, anchors = getsentencelists(df,cols)
    ''' 
    res={}  
    for col in cols:
        res[col]=[]
    for _,row in tqdm(df.iterrows()):
        for col in cols:
            res[col].append(row[col])
    return (res[col] for col in cols)

cols=['positive','anchor']
positives, anchors = getsentencelists(df,cols)


35258it [00:03, 11085.55it/s]

CPU times: user 3.16 s, sys: 19.4 ms, total: 3.18 s
Wall time: 3.19 s


## the positive column has a lot of repeats in it

In [131]:
df['positive'].nunique()

3346

### generate embeddings

In [132]:
%%time

import logging
import torch
import numpy as np

from sentence_transformers import LoggingHandler, SentenceTransformer

#### Just some code to print debug information to stdout
np.set_printoptions(threshold=100)

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO, handlers=[LoggingHandler()]
)
#### /print debug information to stdout


# Load pre-trained Sentence Transformer Model. It will be downloaded automatically
model = SentenceTransformer("all-MiniLM-L6-v2",device="cuda:0" if torch.cuda.is_available() else "cpu",)

# Use "convert_to_tensor=True" to keep the tensors on GPU (if available)
positive_embeddings = model.encode(positives, convert_to_tensor=True)
anchor_embeddings = model.encode(anchors, convert_to_tensor=True)


2024-07-10 16:12:41 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 1102/1102 [00:10<00:00, 103.61it/s]

CPU times: user 25min 31s, sys: 38.6 s, total: 26min 9s
Wall time: 27.9 s


### get similarity score matrix

In [133]:
# We use cosine-similarity 
scores=model.similarity(anchor_embeddings, positive_embeddings)

#to save memory do the following
#del positive_embeddings, anchor_embeddings
# ut.clean_up()

### Hard negative mine the positives for similar positives

This assummes the model has been fine tuned on the dataset first

<mark> Should I take absolute value of cosign similarity so it goes from 0-1?

In [134]:
#the following runs on GPU
import torch
from itertools import compress
from tqdm.auto import tqdm
from numba import njit
import numpy as np

def get_scores_processed(scores, high=0.65):
    """
    Process the scores and generate a mask based on a threshold.

    Parameters:
    scores (torch.Tensor): The input scores.
    high (float, optional): The threshold value. Defaults to 0.65.

    Returns:
    tuple: A tuple containing two numpy arrays - the processed scores and the mask.

    """
    
    # Get the entries below high
    scores_mask = (scores < high).to(scores.device)

    # Set diagonal to False (don't want true positive included in the hard negatives)
    mask = (torch.eye(scores_mask.shape[0], scores_mask.shape[0]) > 0).to(scores.device)
    scores_mask.masked_fill_(mask, False)
   
    return (scores.cpu().detach().numpy(), scores_mask.cpu().detach().numpy())

#the following is compiled into c and runs on a cpu
@njit
def get_results(candidates, scores, true_positive, low=0.5, topn=20):
    """
    Returns a list of results based on the given candidates, scores, true positive, and optional parameters.

    Parameters:
    candidates (list): A list of candidate positions.
    scores (list): A list of cosign similarity scores corresponding to the candidates.
    true_positive (str): The true positive string.
    low (float, optional): The threshold score value. Defaults to 0.5.
    topn (int, optional): The maximum number of results to return. Defaults to 20.

    Returns:
    list: A list of results that meet the criteria.

    """
    res = [pos for score, pos in candidates if score >= low]

    # Remove all duplicate entries
    res = list(set(res))

    # Remove all hits that are the same as the true positive
    res = [val for val in res if val != true_positive]    
    return res[:topn]

@njit
def get_hard_negatives_CPU(scores,scores_mask,positives,low=0.5, high=0.65, topn=20):
    """
    Train a sentencetransformer model, get its average similarity score, use range around that average for hard
    negatives
    expects scores to be nxn matrix of similarity scores, nparray
    expects positives to be a list of n strings
    expects high to be floats denoting the max acceptable similarity score
    topn: int, number of hard negatives to return from torch.top_k
    Get pairs of indices with low<= score <= high
    returns: list of list of positives whose similarity score is between low and high
    """
    
      # use scores_mask to select hard negatives
    hard_negatives=[]
    hn_found=0
    poor_hn_found=0

    for i,row in enumerate(scores_mask):
        #get all the positives and their scores
        # #make sure none of these positives are the same as the true positive
        # #this happens when you derive multiple queries from the same positive
        canditates=[(scores[i,j],positives[j]) for j in range(len(row)) if row[j]==True]

        #sort it by score
        true_positive=positives[i]
        canditates.sort(key=lambda x: x[0],reverse=True)

        res=get_results(canditates, scores, true_positive, low,topn)

        if(len(res)==0):
            #nothing found between high and low, take results from -1 ->low (its -1 because its cosign similarity)
            res=get_results(canditates, scores, true_positive, low=-1,topn=1)
            poor_hn_found+=1
        else:
            hn_found+=1
        
        hard_negatives.append(res)
    print(f"hn_found={hn_found}, poor_hn_found={poor_hn_found}")
    return hard_negatives


In [135]:
high=0.65
ghn3=get_hard_negatives_CPU(*(get_scores_processed(scores,high)),positives,high=high)
# ghn3[:5]

In [112]:
# small test case for above
# ns=50
# high=0.65
# scores2=scores[:ns,:ns]
# ghn3=get_hard_negatives_CPU(*(get_scores_processed(scores2,high)),positives[:ns],high=high)

In [ ]:
#lets see how they look
# for p in ghn[:20]: print(len(p))
def showrow(anchors, positives,ghn3,row=0):
    print(f'ANCHOR={anchors[row]}')
    print(f'POSITIVE={positives[row]}')
    print(f'HARDNEG={ghn3[row][0]}')
    print()
showrow(anchors, positives,ghn3,row=0)
showrow(anchors, positives,ghn3,row=1)
showrow(anchors, positives,ghn3,row=2)

ANCHOR=what methods does a ride-sharing platform employ to ascertain the whereabouts of a customer when they are using the platform's services for rides or parcel dispatch?
POSITIVE=information we collect through your use of our services when you use our services, we collect information about you in the following general categories: location information: when you use the services for transportation or delivery, we collect precise location data about the trip from the uber app used by the driver. if you permit the uber app to access location services through the permission system used by your mobile operating system ("platform"), we may also collect the precise location of your device when the app is running in the foreground or background. we may also derive your approximate location from your ip address.
HARDNEG=personal information to be collected during linkage process to the membership, the company collects personal information such as names, sex, profile images, locations, email a

### add to dataframe and save to json

In [ ]:
try:
    df.drop(columns=['most_dissimilar_context','id'],inplace=True)
    df.drop(columns=['hardnegatives','hardnegatives_indexed','hardnegatives_reverse_indexed'],inplace=True)
except:
    pass

df.head()

,anchor,positive
15,what methods does a ride-sharing platform empl...,information we collect through your use of our...
115,do the headings have any precedential value?,9.11 headings. headings used in this agreement...
161,what types of information are considered 'shar...,ii. information we collect in providing our se...
200,what mechanisms do independent parties utilize...,how the information about our users of kids ap...
223,what protocols are in place for gathering pers...,any of your information you provide or permit ...


In [ ]:
df.columns
df['hardnegatives']=ghn3
df.reset_index(drop=True,inplace=True) #make sure the indexes are continuous
df.head()

/tmp/ipykernel_133590/2765484543.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['hardnegatives']=ghn3


,anchor,positive,hardnegatives
0,what methods does a ride-sharing platform empl...,information we collect through your use of our...,[personal information to be collected during l...
1,do the headings have any precedential value?,9.11 headings. headings used in this agreement...,[5.1 representations and warranties of spinrec...
2,what types of information are considered 'shar...,ii. information we collect in providing our se...,[personal information to be collected during l...
3,what mechanisms do independent parties utilize...,how the information about our users of kids ap...,[the purpose for such collection: tabtale allo...
4,what protocols are in place for gathering pers...,any of your information you provide or permit ...,[personal information to be collected during l...


In [ ]:
df[df['hardnegatives'].apply(lambda x: len(x) == 0)]

,anchor,positive,hardnegatives
23,will users be compensated for the use of their...,received information shared with third parties...,[]
32,is it possible for the company to distribute d...,received information shared with third parties...,[]
68,will the company receive compensation for shar...,received information shared with third parties...,[]
86,can the organization transmit anonymized user ...,received information shared with third parties...,[]
87,are age details necessary for all promotions p...,"read more we may sponsor contests, challenges,...",[]
110,are users entitled to any compensation for the...,received information shared with third parties...,[]
143,is age verification necessary for participatio...,"read more we may sponsor contests, challenges,...",[]
151,can the organization transmit anonymized clien...,received information shared with third parties...,[]
160,will the company receive any form of compensat...,received information shared with third parties...,[]
165,will users be compensated?,received information shared with third parties...,[]


In [ ]:
#save to disk
df.to_json('../data/trn_with_hard_negatives.json')

### The above column will blow up the size of trn_with_hard_negatives.json, use an indexing scheme instead?

In [ ]:
import os
import json

class index_values():
    def __init__(self, filename):
        # self.unique_positives=list(set(positives))
        self.filename=filename  #used to save this indexer object
        self.__load()

    def getval(self,index):
        return self.indexer_reverse[index]
    
    def getindex(self,val):
        return self.indexer[val]

    def set(self,strings):
        unique_strings=list(set(strings))
        self.indexer={val:index for index,val in enumerate(unique_strings)}
        self.indexer_reverse={index:val for val,index in self.indexer.items()}
        self.__store()

    def __load(self):
        self.indexer={}
        self.indexer_reverse={}
        
        if os.path.exists(self.filename):
            with open(self.filename) as f:
                self.indexer=json.load(f)
                self.indexer_reverse={index:val for val,index in self.indexer.items()}

    def __store(self):
        with open(self.filename,'w') as f:
            json.dump(self.indexer,f)
        

In [ ]:
indexer=index_values('../data/indexer.json')
indexer.set(positives)
# indexer.getval(0)
# indexer.getindex('2. the contractor shall not, without the state’s prior written consent, copy, disclose, publish, release, transfer, disseminate, use, or allow access for any purpose or in any form, any confidential information except for the sole and exclusive purpose of performing under the contract.')


# unique_pos=list(set(positives))
# indexer={positive:index for index,positive in enumerate(unique_pos)}
# indexer_reverse={index:positive for positive,index in indexer.items()}

In [ ]:
def indx_row(row,col,indxr):
    l=[]
    for positive in row[col]:
        l.append( indxr[positive])
    return l

# df1=df.head()
df['hardnegatives_indexed']=df.apply(indx_row,args=('hardnegatives',indexer.indexer),axis=1)
df['hardnegatives_reverse_indexed']=df.apply(indx_row,args=('hardnegatives_indexed',indexer.indexer_reverse),axis=1)

# df1['hardnegatives_reversed']=df1.apply(indx_row,axis=1)

In [ ]:
len(df)
df.tail()

,anchor,positive,hardnegatives,hardnegatives_indexed,hardnegatives_reverse_indexed
35253,for what duration must the verticalnet branded...,2.1. hosting and maintenance. leadersonline sh...,[2.3. link license. verticalnet hereby grants ...,"[1854, 1295]",[2.3. link license. verticalnet hereby grants ...
35254,who controls third-party cookies?,it is understood that this policy covers the u...,[cookies we have the right to set and access c...,"[323, 1698]",[cookies we have the right to set and access c...
35255,detail 'information' storage measures.,ii. information we collect in providing our se...,[],[],[]
35256,what does google expressly warrant regarding t...,8.2 google warrants that the distribution prod...,[2.2 third party distribution. distributor may...,"[2424, 579, 2999]",[2.2 third party distribution. distributor may...
35257,what are the consequences of not updating or p...,a11: bandai namco may amend this privacy polic...,[we will hold all data gathered in strict conf...,[1375],[we will hold all data gathered in strict conf...


In [ ]:
# df=df[df['hardnegatives'].apply(lambda x: len(x) == 0)]
# df.drop(columns=['hardnegatives','hardnegatives_indexed','hardnegatives_reverse_indexed'],inplace=True)
# df

KeyError: 'hardnegatives'

In [ ]:
# df[df['hardnegatives'].apply(lambda x: len(x) == 0)]

,anchor,positive,hardnegatives
23,will users be compensated for the use of their...,received information shared with third parties...,[]
32,is it possible for the company to distribute d...,received information shared with third parties...,[]
68,will the company receive compensation for shar...,received information shared with third parties...,[]
86,can the organization transmit anonymized user ...,received information shared with third parties...,[]
87,are age details necessary for all promotions p...,"read more we may sponsor contests, challenges,...",[]
110,are users entitled to any compensation for the...,received information shared with third parties...,[]
143,is age verification necessary for participatio...,"read more we may sponsor contests, challenges,...",[]
151,can the organization transmit anonymized clien...,received information shared with third parties...,[]
160,will the company receive any form of compensat...,received information shared with third parties...,[]
165,will users be compensated?,received information shared with third parties...,[]


In [77]:
scores[15,:]

tensor([0.4512, 0.1724, 0.2356,  ..., 0.2604, 0.1853, 0.1980], device='cuda:0')

In [27]:
print(df1.iloc[0,2])
print(df1.iloc[0,4])

['if a party or any third party to whom such party has provided confidential information becomes legally compelled (by oral question, deposition, interrogatory, request for documents, subpoena, civil investigative demand or similar process or by rule, regulation or other applicable law) to disclose any confidential information, such party shall promptly notify the other party of such requirement before any disclosure is made so that the other party may seek a protective order or other appropriate remedy or may waive compliance with the terms of this agreement.', '2.5 neither party shall be required to keep confidential any information which is, or becomes, publicly available, is independently developed by either party outside the scope of this agreement, or is rightfully obtained from third parties.', '3. confidentiality the parties hereto agree that each shall treat confidentially the terms and conditions of this agreement and all information provided by each party to the other regard

# Junk

In [ ]:
# %%time
# # Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
# top_k = min(100, len(positive_embeddings))

# #for just 1 query
# anchor=anchor_embeddings[0]
# # We use cosine-similarity and torch.topk to find the highest 5 scores
# similarity_scores = model.similarity(anchor, positive_embeddings)[0]
# scores, indices = torch.topk(similarity_scores, k=top_k)

# gt90=0
# gt80=0
# gt70=0
# lt70=0
# for anchor in anchor_embeddings:
#     # We use cosine-similarity and torch.topk to find the highest 5 scores
#     similarity_scores = model.similarity(anchor, positive_embeddings)[0]
#     scores, indices = torch.topk(similarity_scores, k=top_k)
#     if scores[0]>90: 
#         gt90+=1 
#     elif scores[0]>80: 
#         gt80+=1 
#     elif scores[0]>70: 
#         gt70+=1 
#     else: 
#         lt70+=1
# print(f"gt90: {gt90}, gt80: {gt80}, gt70: {gt70}, lt70: {lt70}")
    


gt90: 0, gt80: 0, gt70: 0, lt70: 35258
CPU times: user 14.1 s, sys: 10.3 ms, total: 14.1 s
Wall time: 14.2 s
